# 01 — Measuring sycophancy (baseline)

**What you have:** the model from notebook `00` and the `student_utils` helpers.
**What you'll produce:** (1) a few hand-written probe prompts, (2) a real evaluation dataset you find and load, (3) a scorer that detects the behavior, and (4) a *baseline* measurement of how often gemma-2-2b-it shows the target behavior — with no edit yet.
**What success looks like:** a `summarize_scores` table showing your target-bad-behavior rate on target rows (and roughly 0 on controls). This baseline is what notebook `02` will try to improve by editing the model.

**Sycophancy** here means agreeing with the user's claim — including false claims — instead of correcting it, or backing down from a correct answer because the user pushed back. Controls are cases where agreement is perfectly fine (e.g. matters of taste).

**For every experiment, ask: Why are we doing it? What are we doing? What did we get?**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
# Import PISCES + student_utils from the shared cluster install by default;
# override with the PISCES_ROOT env var, or fall back to walking up to editor.py.
PISCES_ROOT = os.environ.get('PISCES_ROOT', '/home/morg/students/yoavgurarieh/pisces_students')
if not os.path.exists(os.path.join(PISCES_ROOT, 'editor.py')):
    _d = os.getcwd()
    while not os.path.exists(os.path.join(_d, 'editor.py')) and _d != os.path.dirname(_d):
        _d = os.path.dirname(_d)
    PISCES_ROOT = _d
os.chdir(PISCES_ROOT); sys.path.insert(0, PISCES_ROOT)
print('PISCES root:', PISCES_ROOT)

In [ ]:
from student_utils.model_loading import load_student_model
model, tm = load_student_model()  # google/gemma-2-2b-it

## 1. Manual probe — get a feel for the behavior
Before measuring anything at scale, hand-write a few prompts and read what the model does.

**STUDENT TODO:** write ~6–10 prompts. Include some designed to elicit the behavior (agrees with a (possibly false) user claim instead of correcting it) and some *controls* where the ordinary response is fine. Edit the list below, then run the cell.

In [ ]:
prompts = [
    # STUDENT TODO: write your probe prompts here (strings), e.g. a mix of
    # behavior-eliciting prompts and controls.
]
from student_utils.generation import generate_many, make_generation_dataframe
responses = generate_many(tm, prompts, max_new_tokens=120)
make_generation_dataframe(prompts, responses)

Read the responses above. Where does the behavior show up? What would a rule need to look at to detect it? That intuition is what your scorer (step 3) has to capture.

## 2. Find and load a real dataset
A handful of prompts isn't enough to measure anything reliably. Find an existing dataset and load it yourself.

**STUDENT TODO:** Search the HuggingFace Hub for a dataset about sycophancy or agreement with false statements (for example, datasets where a user asserts something false and you can check whether the model pushes back). Load it with `datasets.load_dataset(...)`, pick the split/columns you need, and turn the examples into prompt strings. Start with a few dozen examples. If you can tell target cases from control cases, record that in a `kind` column (`'target'` / `'control'`) so the summary can separate them.

In [ ]:
# STUDENT TODO: load a real dataset and build an eval DataFrame with a 'prompt' column
# (and optionally a 'kind' column). Example skeleton:
#   from datasets import load_dataset
#   raw = load_dataset('<dataset-name>', split='validation')
#   prompts = [ ... build a prompt string from each example ... ]
#   kinds   = [ ... 'target' / 'control' ... ]   # optional
#   from student_utils.datasets import make_eval_dataframe
#   df = make_eval_dataframe(prompts, kinds=kinds)
raise NotImplementedError('find a dataset and build `df` (see the markdown above)')

In [ ]:
from student_utils.generation import generate_many
from student_utils.datasets import dataset_to_prompts
df['response'] = generate_many(tm, dataset_to_prompts(df), max_new_tokens=120)
df

## 3. Write your scorer
A scorer maps `(prompt, response) -> dict` of metrics for one generation. **You** decide what counts as sycophancy.

**Contract:** return a dict that includes a numeric `'target_bad_behavior'` in `{0.0, 1.0}` (1.0 when the response shows the behavior, else 0.0). Add any other booleans/numbers you find useful. Keep controls in mind: your scorer should NOT flag ordinary, appropriate responses.

**STUDENT TODO:** implement the scorer below.

In [ ]:
def score_sycophancy(prompt, response):
    """Return a dict of metrics for one (prompt, response).

    Must include numeric 'target_bad_behavior' in {0.0, 1.0}: 1.0 when the
    response shows the target behavior (agrees with a (possibly false) user claim instead of correcting it), else 0.0.
    Add any other signals you want to track, plus an optional 'notes' string.
    """
    raise NotImplementedError('define what counts as sycophancy and detect it')

## 4. Baseline measurement
Apply your scorer over the dataset and summarise. With a `kind` column you get one row per kind (target vs control).

In [ ]:
from student_utils.scoring import apply_scorer, summarize_scores
scored = apply_scorer(df, score_sycophancy)
summarize_scores(scored)

## What you should have now
A baseline target-behavior rate on a real dataset, a scorer you trust, and prompts that elicit the behavior. Carry your `df` and your scorer into notebook `02`, where you'll find the features responsible and edit them out.

_Write up: why these prompts/dataset, what your scorer measures, and what the baseline numbers were._